In [17]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA
import os


# Load PDF
loader = PyPDFLoader("AIAgents.pdf")
documents = loader.load()

# Split text
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

# Embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Store in ChromaDB
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

# Save database
vectorstore.persist()

print("PDF processed successfully!") 


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4283.98it/s]


PDF processed successfully!


In [18]:
# LLM
llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="qwen/qwen3-32b"
) 

retriever = vectorstore.as_retriever( search_kwargs={"k": 3})

query = "what is AI Agents?"

# Retrieve top chunks
docs = retriever.invoke(query)

print("Retrieved Chunks:\n")

for i, doc in enumerate(docs):
    print(f"\nChunk {i+1}")
    print(doc.page_content)

# Send to LLM
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff"
)

response = qa_chain.invoke({"query": query})

print("\nFinal Answer:\n")
print(response["result"])

Retrieved Chunks:


Chunk 1
explores the principles, design, and operations of these agents, highlighting their potential 
applications 
 
1.1. What are AI Agents 
 
At its core, a Generative AI agent is an application designed to achieve a specific objective by 
perceiving its environment and taking actions based on the tools available. These agents are 
autonomous, capable of operating independently without direct human oversight when provided 
with clear goals or objectives. Agents can reason for subsequent actions to move toward their 
ultimate objective even without explicit human instructions. 
 
While the concept of agents in artificial intelligence is broad and versatile, this white paper 
focuses on the types of agents that can be constructed using contemporary Generative AI models. 
To understand the functionality of these agents, it is important to examine the foundational 
elements that govern their behaviour, actions, and decision -making processes. These elements

Chunk 2

In [20]:
print("\nFinal Answer:\n")
print(response["result"])


Final Answer:

<think>
Okay, the user is asking, "What is AI agents?" I need to use the provided context to answer this.

Looking at the context, there's a section titled "1.1. What are AI Agents" that explains Generative AI agents. The key points are that they're applications designed to achieve specific objectives by perceiving their environment and taking actions using available tools. They are autonomous, operate independently without human oversight when given clear goals, and can reason about subsequent actions to reach their objectives without explicit instructions.

The context mentions that while AI agents are a broad concept, the focus here is on those built with contemporary Generative AI models. It also talks about foundational elements governing their behavior, actions, and decision-making processes.

I should define AI agents based on this, highlighting autonomy, goal-oriented behavior, use of tools, and reasoning capabilities. I need to make sure not to add any informat